# 08 EDA 模块 (core.eda) 完整示例

覆盖数据概览 / 目标变量分析 / 特征关系 / 相关性 / 稳定性 / Vintage / 综合报告。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import hscredit as hsc
from hscredit import germancredit, init_setting
from hscredit import (
    data_info, missing_analysis, feature_summary, numeric_summary,
    category_summary, data_quality_report,
    target_distribution, bad_rate_overall, bad_rate_by_dimension,
    bad_rate_trend, bad_rate_by_bins, sample_distribution,
    iv_analysis, batch_iv_analysis, woe_analysis, binning_bad_rate,
    correlation_matrix, high_correlation_pairs, vif_analysis,
    psi_analysis, batch_psi_analysis, csi_analysis,
    vintage_analysis, eda_summary,
)
init_setting()
df = germancredit()
target = 'creditability'
np.random.seed(42)
start = pd.Timestamp('2023-01-01')
df['apply_date'] = [start + pd.Timedelta(days=int(d)) for d in np.random.randint(0,365,len(df))]
print(df.shape)

## 1. 数据概览

In [ ]:
print(data_info(df))

In [ ]:
missing_analysis(df).head(10)

In [ ]:
num_feats = df.select_dtypes('number').columns.drop(target).tolist()
feature_summary(df[num_feats + [target]], y=target).head(8)

In [ ]:
numeric_summary(df[num_feats]).head(6)

In [ ]:
cat_feats = df.select_dtypes('object').columns.tolist()
category_summary(df[cat_feats]).head(6)

In [ ]:
data_quality_report(df).head()

## 2. 目标变量分析

In [ ]:
print('总坏率:', bad_rate_overall(df, target))

In [ ]:
bad_rate_by_dimension(df, target, dim_col='purpose').head(8)

In [ ]:
bad_rate_trend(df, target, date_col='apply_date', freq='M')

In [ ]:
bad_rate_by_bins(df, feature='duration.in.month', target=target, n_bins=8)

In [ ]:
sample_distribution(df, date_col='apply_date', target=target, freq='M')

## 3. 特征与目标关系 (IV / WOE)

In [ ]:
iv_result = iv_analysis(df, feature='duration.in.month', target=target, n_bins=8)
print(f'IV: {iv_result["IV值"]}')
iv_result['分箱明细'].head()

In [ ]:
batch_iv = batch_iv_analysis(df, features=num_feats, target=target)
batch_iv.sort_values('IV值', ascending=False).head(10)

In [ ]:
woe_result = woe_analysis(df, feature='credit.amount', target=target, n_bins=6)
woe_result['WOE明细']

In [ ]:
binning_bad_rate(df, feature='age.in.years', target=target, n_bins=5)

## 4. 相关性分析

In [ ]:
corr_df = correlation_matrix(df[num_feats[:8]])
corr_df.head(6)

In [ ]:
high_corr = high_correlation_pairs(df[num_feats[:8]], threshold=0.5)
high_corr

In [ ]:
vif_df = vif_analysis(df[num_feats[:6]])
vif_df

## 5. 稳定性分析 (PSI / CSI)

In [ ]:
from sklearn.model_selection import train_test_split
df_tr, df_te = train_test_split(df, test_size=0.3, random_state=42)
psi_result = psi_analysis(df_tr, df_te, feature='duration.in.month')
print(f'PSI: {psi_result["PSI值"]}, 稳定性: {psi_result["稳定性"]}')
psi_result['分箱明细']

In [ ]:
df['apply_month'] = df['apply_date'].dt.to_period('M').astype(str)
periods = sorted(df['apply_month'].unique())
batch_psi_df = batch_psi_analysis(df, num_feats[:4], 'apply_month',
    base_period=periods[0], compare_periods=periods[1:4])
batch_psi_df

## 6. Vintage 分析

In [ ]:
np.random.seed(42)
df_vint = pd.DataFrame({
    'apply_month': pd.date_range('2022-01', periods=400, freq='7D').to_period('M').astype(str),
    'mob': np.random.randint(0, 13, 400),
    'is_bad': np.random.binomial(1, 0.1, 400),
    'amount': np.random.uniform(1000, 50000, 400),
})
vint_result = vintage_analysis(df_vint, cohort_col='apply_month', mob_col='mob', target='is_bad')
vint_result.head()

## 7. eda_summary — 综合报告

In [ ]:
summary = eda_summary(df, target=target, features=num_feats[:6])
print(type(summary))
for k, v in summary.items():
    print(f'  {k}: {type(v).__name__}')